# Certamen de laboratorio · Minería de Datos · Unidad II · Sección 2

**martes 27 de octubre de 2026** · **individual** · 120 minutos · 100 puntos · 40 % de la nota de práctica

**Nombre:** _escriba aquí su nombre_ · **RUT:** _escriba aquí su RUT_

## Antes de empezar (lea esto primero)

1. **Archivo → Guardar una copia en Drive.** Trabaje siempre en su copia.
2. Ejecute la **celda ▶ entregada** (la primera de código). Debe ver: *Datos cargados: 220 socios (170 de entrenamiento y 50 de prueba), 320 pedidos, 36 semanas y 57 estudiantes.*
3. Cada tarea tiene siempre las mismas partes, en este orden:

| Ícono | Qué es | Qué hace usted |
|:-:|---|---|
| 📖 | **Contexto** | Entender el problema del negocio |
| 💡 | **Concepto clave** | Repasar la idea que hay detrás |
| 🎯 | **Qué hacer** | Seguir los pasos |
| 📦 🔍 | **Qué va a obtener y qué representa** | Saber de antemano qué resultado buscar y qué significa para el caso |
| 🧩 | **Función con su firma** | Completar **solo el cuerpo**. No cambie el nombre ni los parámetros. El docstring dice qué recibe, qué devuelve y trae un ejemplo |
| 🧪 | **Prueba** con un ejemplo mínimo | Ejecutarla. Debe dar **exactamente** el resultado indicado, que se explica al lado. Si no da, corrija la función antes de seguir |
| 📊 | **Análisis** con los datos del laboratorio | Escribir el código que pide la tarea |
| ✅ | **Resultado esperado** | Comparar: dice **qué debe mostrar** su celda 📊 y en qué formato |
| ✍️ | **Respuesta** | Responder con **sus propios números** |

4. **De los datos al conocimiento.** Cada tarea termina con una pregunta 🧠. El curso trabaja con una escalera de tres peldaños:

| Peldaño | Qué es | Ejemplo en el gimnasio |
|---|---|---|
| **Dato** | Un hecho suelto | «El socio 2023 viene a veces (frecuencia media), entrena solo y no renovó» |
| **Información** | Datos procesados que muestran un patrón | «El árbol acierta 9 de cada 10 socios de prueba» |
| **Conocimiento** | Información convertida en una **decisión** | «Hay que invitar a clases grupales a quienes vienen a veces y entrenan solos» |

   Su código produce **información**; sus respuestas 🧠 deben llegar al **conocimiento**.

5. Puede usar **todos los recursos** (apuntes, guías, internet, IA). Pero se evalúa que **interprete sus resultados**: una respuesta genérica, sin sus números, no tiene puntaje.
6. Tiempo sugerido: **Parte I** hasta el minuto 40, **Parte II** hasta el 80 y **Parte III** hasta el 120.
7. Al terminar: **Archivo → Descargar → Descargar .ipynb**, y súbalo al **Aula Virtual** como `Apellido_Nombre.ipynb`.

| Parte | Caso | Tareas | Puntos |
|---|---|:-:|:-:|
| I | Gimnasio: ZeroR, OneR y árbol | 1 a 3 | 35 |
| II | Librería online: soporte, confianza, lift y regresión | 4 a 6 | 35 |
| III | Preuniversitario online: k-means por rondas y sus gráficas | 7 y 8 | 30 |

In [ ]:
# ▶ CELDA ENTREGADA — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/certamen-lab-s2/"
gimnasio = pd.read_csv(RUTA + "lab_gimnasio.csv")            # Parte I
pedidos = pd.read_csv(RUTA + "lab_pedidos.csv")              # Parte II (Apriori)
semanas = pd.read_csv(RUTA + "lab_semanas.csv")              # Parte II (regresión)
preu = pd.read_csv(RUTA + "lab_preu.csv")                    # Parte III

entrenamiento = gimnasio[gimnasio["Conjunto"] == "entrenamiento"]
prueba = gimnasio[gimnasio["Conjunto"] == "prueba"]
canastas = pedidos.groupby("Pedido")["Producto"].apply(set).tolist()          # un conjunto por pedido
puntos = list(zip(preu["Video"], preu["Ejercicios"]))                        # pares (x, y)

ATRIBUTOS = ["Jornada", "Frecuencia", "Clases"]
PRODUCTOS = ['Témpera', 'Pincel', 'Block', 'Lápiz', 'Cuaderno', 'Compás']


def acierto(predicciones):
    """Proporción de socios de PRUEBA bien predichos (entre 0 y 1). Recibe una predicción por socio de prueba."""
    return (prueba["Renovo"].values == pd.Series(predicciones).values).mean()

ESTRELLAS_INICIO = [(1, 1), (9, 2), (8, 3)]

print(f"Datos cargados: {len(gimnasio)} socios ({len(entrenamiento)} de entrenamiento y {len(prueba)} de prueba), "
      f"{len(canastas)} pedidos, {len(semanas)} semanas y {len(puntos)} estudiantes.")

---
# Parte I · Gimnasio (35 puntos)

El gimnasio quiere predecir **qué socios no van a renovar su plan**, para contactarlos antes. La tabla `gimnasio` tiene **220 socios** con:

| Columna | Valores |
|---|---|
| `Socio` | número de socio (2001 a 2220) |
| `Jornada` | Mañana · Tarde (cuándo suele entrenar) |
| `Frecuencia` | Alta · Media · Baja (cuántas veces viene a la semana) |
| `Clases` | Grupales · Solo (si va a clases grupales o entrena por su cuenta) |
| `Plan` | Mensual · Semestral · Anual (no se usa en las tareas) |
| `Renovo` | **Sí · No** ← la clase que se quiere predecir |
| `Conjunto` | `entrenamiento` (170 socios, para **construir** los modelos) · `prueba` (50 socios, para **evaluarlos**) |

Ya están separados en las variables **`entrenamiento`** y **`prueba`**. Los modelos se construyen **solo** con `entrenamiento`.

## Tarea 1 · ZeroR: la línea base (8 puntos)

**📖 Contexto.** Antes de usar un modelo, el gimnasio quiere saber cuánto acierta **sin analizar nada**.

> **💡 Concepto clave.** **ZeroR** predice siempre la clase que más se repite. Su acierto es la **línea base**: el mínimo que cualquier modelo debe superar.

### 🎯 Qué hacer

1. Complete `zeror()` siguiendo las pistas que trae adentro y ejecute la prueba 🧪.
2. Complete los `___` de la celda 📊.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Lo que predice ZeroR, sus **errores en entrenamiento** y su **acierto en prueba** (un número entre 0 y 1). | Ese acierto es **la vara para medir** los demás modelos: lo que se logra sin analizar ningún dato. |

In [ ]:
# 🧩 Tarea 1 · complete el cuerpo de la(s) función(es)
def zeror(df, clase):
    """ZeroR: predice SIEMPRE la clase que más se repite.

    Recibe:   df    → la tabla (DataFrame)
              clase → el nombre de la columna que se quiere predecir, aquí "Renovo"
    Devuelve: (clase_mayoritaria, errores)
              errores = cuántas filas NO son de la clase mayoritaria (ahí ZeroR se equivoca)
    Ejemplo:  si hay 4 filas «Sí» y 7 «No»  →  ("No", 4)
    """
    # Pista 1: cuente cuántas filas hay de cada clase   →  conteo = df[clase].value_counts()
    #          (queda ordenado de mayor a menor)
    # Pista 2: la clase mayoritaria es la primera          →  conteo.index[0]
    # Pista 3: los errores son todas las demás filas       →  len(df) - conteo.iloc[0]
    # Pista 4: devuelva las dos cosas                      →  return mayoritaria, errores
    pass

In [ ]:
# 🧪 Prueba de la tarea 1 con un ejemplo mínimo: ejecútela sin cambiarla
mini = pd.DataFrame({
    "Frecuencia": ["Alta",     "Alta", "Media", "Media",    "Media", "Baja",     "Baja", "Baja", "Alta"],
    "Clases":     ["Grupales", "Solo", "Solo",  "Grupales", "Solo",  "Grupales", "Solo", "Solo", "Solo"],
    "Renovo":     ["Sí",       "Sí",   "No",    "Sí",       "No",    "No",       "No",   "No",   "Sí"]})
c, e = zeror(mini, "Renovo")
print(c, e)
# ✅ Debe mostrar: No 4

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`No 4`**: en los 9 socios del ejemplo hay 5 «No» y 4 «Sí»: ZeroR predice «No» y falla en los 4 «Sí». Si le da 5, está contando aciertos.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 1 · complete los ___ y ejecute
clase_zr, errores_zr = zeror(___, "Renovo")           # use los socios de entrenamiento
print("ZeroR predice:", clase_zr, "| errores:", errores_zr)
acierto_zr = acierto([clase_zr] * len(prueba))          # ZeroR predice lo mismo para los 50 socios de prueba
print("Acierto en prueba:", round(acierto_zr, 3))

**✅ Compare:** su celda 📊 debe mostrar lo que predice ZeroR, sus **errores en entrenamiento** y su **acierto en prueba** (un número entre 0 y 1).

### ✍️ Responda

**1. Sus resultados:** ¿Qué predice ZeroR y qué acierto obtiene en prueba?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Si el gimnasio supusiera que **todos renuevan**, ¿en cuántos de cada 10 socios se equivocaría?

_Su respuesta:_

## Tarea 2 · OneR: ¿qué dato explica más la renovación? (12 puntos)

**📖 Contexto.** El gimnasio quiere saber cuál de tres datos (jornada, frecuencia de asistencia o tipo de clases) **explica mejor** si el socio renueva.

> **💡 Concepto clave.** **OneR** prueba cada dato por separado: para cada valor predice la clase más frecuente, y cuenta los errores. Gana el dato con **menos errores**.

### 🎯 Qué hacer

1. Complete `oner()` siguiendo las pistas (puede usar su `zeror()`) y ejecute la prueba 🧪.
2. Complete los `___` de la celda 📊: errores de cada atributo, el ganador y su acierto en prueba.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Los **errores de los 3 atributos**, el **ganador** con **su regla** (un valor → una clase) y su **acierto en prueba**. | El ganador es **el dato que más ayuda a predecir** la renovación; su regla ya es una política simple para el gimnasio. |

In [ ]:
# 🧩 Tarea 2 · complete el cuerpo de la(s) función(es)
def oner(df, atributo, clase):
    """OneR con UN atributo: una regla por cada valor del atributo.

    Para cada valor del atributo: la regla es la clase que más se repite en ese valor, y los
    errores son las filas de ese valor que tienen la otra clase. Se suman los errores.
    Recibe:   df, atributo (p. ej. "Frecuencia"), clase ("Renovo")
    Devuelve: (reglas, errores)
              reglas = diccionario {valor: clase predicha}
    Ejemplo:  si con "Baja" hay 1 «Sí» y 3 «No», su regla es "Baja" → "No" y suma 1 error.
              Devuelve algo como ({"Alta": "Sí", "Baja": "No", "Media": "Sí"}, 1)
    """
    reglas = {}
    errores = 0
    for valor, grupo in df.groupby(atributo):     # un grupo de filas por cada valor del atributo
        # Pista: zeror(grupo, clase) le da la clase mayoritaria del grupo y sus errores
        # 1. guarde la regla de ese valor:   reglas[valor] = ...
        # 2. sume los errores de ese valor:  errores = errores + ...
        pass
    return reglas, errores

In [ ]:
# 🧪 Prueba de la tarea 2 con un ejemplo mínimo: ejecútela sin cambiarla
mini = pd.DataFrame({
    "Frecuencia": ["Alta",     "Alta", "Media", "Media",    "Media", "Baja",     "Baja", "Baja", "Alta"],
    "Clases":     ["Grupales", "Solo", "Solo",  "Grupales", "Solo",  "Grupales", "Solo", "Solo", "Solo"],
    "Renovo":     ["Sí",       "Sí",   "No",    "Sí",       "No",    "No",       "No",   "No",   "Sí"]})
reglas, e = oner(mini, "Frecuencia", "Renovo")
print(dict(sorted(reglas.items())), e)
# ✅ Debe mostrar: {'Alta': 'Sí', 'Baja': 'No', 'Media': 'No'} 1

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`{'Alta': 'Sí', 'Baja': 'No', 'Media': 'No'} 1`**: Alta: 3 Sí (regla Sí, 0 errores); Baja: 3 No (regla No, 0 errores); Media: 2 No y 1 Sí (regla No, 1 error). Total: 1 error.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 2 · complete los ___ y ejecute
for atributo in ATRIBUTOS:
    reglas, errores = oner(entrenamiento, ___, "Renovo")
    print(atributo, "→", errores, "errores")
ganador = ___                                            # escriba entre comillas el atributo con MENOS errores
reglas_ganador, _ = oner(entrenamiento, ganador, "Renovo")
print("Regla del ganador:", reglas_ganador)
pred_oner = [reglas_ganador[v] for v in prueba[ganador]]  # la regla aplicada a cada socio de prueba
print("Acierto en prueba:", round(acierto(pred_oner), 3))

**✅ Compare:** su celda 📊 debe mostrar los **errores de los 3 atributos**, el **ganador** con **su regla** (un valor → una clase) y su **acierto en prueba**.

### ✍️ Responda

**1. Sus resultados:** ¿Qué atributo gana, cuál es su regla y cuánto acierta en prueba comparado con ZeroR? ¿Qué atributo no le gana a ZeroR?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Según la regla ganadora, ¿qué le recomendaría al gimnasio para **retener a sus socios**?

_Su respuesta:_

## Tarea 3 · Árbol: una segunda pregunta donde hace falta (15 puntos)

**📖 Contexto.** La regla de OneR se equivoca casi siempre en **una misma rama**. El árbol agrega **otra pregunta solo en esa rama**.

> **💡 Concepto clave.** Una **rama impura** mezcla «Sí» y «No». En ella se busca la segunda pregunta aplicando OneR **solo a los socios de esa rama**. Las demás ramas quedan igual.

### 🎯 Qué hacer

1. No hay función nueva: use `zeror()`, `oner()` y la función entregada `predecir_arbol()`.
2. Complete los `___` de la celda 📊 en tres pasos: (1) la rama con más errores, (2) la segunda pregunta y (3) el acierto del árbol.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Los **errores de cada rama**, la **rama impura**, la **segunda pregunta** y el **acierto del árbol en prueba**. | El árbol es un **protocolo de dos preguntas**. Si acierta más que OneR, combinar dos datos predice mejor que uno solo. |

In [ ]:
# ▶ FUNCIÓN ENTREGADA — no la modifique. Clasifica un socio con el árbol de dos preguntas.
def predecir_arbol(fila, raiz, reglas_raiz, rama, atributo2, reglas2):
    """Si el socio cae en la rama impura, decide la 2.ª pregunta; si no, decide la regla de la raíz.

    Uso: predecir_arbol(fila, ganador, reglas_ganador, rama, atributo2, reglas2)
         con ganador y reglas_ganador de la tarea 2, y rama, atributo2 y reglas2 de esta tarea.
    """
    if fila[raiz] == rama:
        return reglas2.get(fila[atributo2], reglas_raiz[rama])
    return reglas_raiz[fila[raiz]]

In [ ]:
# 🧪 Prueba de la tarea 3 con un ejemplo mínimo: ejecútela sin cambiarla
mini = pd.DataFrame({
    "Frecuencia": ["Alta",     "Alta", "Media", "Media",    "Media", "Baja",     "Baja", "Baja", "Alta"],
    "Clases":     ["Grupales", "Solo", "Solo",  "Grupales", "Solo",  "Grupales", "Solo", "Solo", "Solo"],
    "Renovo":     ["Sí",       "Sí",   "No",    "Sí",       "No",    "No",       "No",   "No",   "Sí"]})
rama_mini = mini[mini["Frecuencia"] == "Media"]
c, e = zeror(rama_mini, "Renovo")
r, e2 = oner(rama_mini, "Clases", "Renovo")
print(c, e, dict(sorted(r.items())), e2)
# ✅ Debe mostrar: No 1 {'Grupales': 'Sí', 'Solo': 'No'} 0

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`No 1 {'Grupales': 'Sí', 'Solo': 'No'} 0`**: en el ejemplo, la rama Media es la impura (2 No y 1 Sí: 1 error). Dentro de ella, Clases separa sin errores: el socio de clases grupales renovó y los 2 que entrenan solos no.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 3 · complete los ___ y ejecute
# (1) Errores de cada rama: la que tiene MÁS errores es la impura
for valor in sorted(entrenamiento[ganador].unique()):
    socios_rama = entrenamiento[entrenamiento[ganador] == valor]
    print(valor, "→", zeror(socios_rama, "Renovo")[1], "errores")
rama = ___                                               # escriba entre comillas la rama con más errores

# (2) Segunda pregunta: OneR solo con los socios de esa rama
socios_rama = entrenamiento[entrenamiento[ganador] == rama]
for atributo in ATRIBUTOS:
    if atributo != ganador:
        print(atributo, "→", oner(___, atributo, "Renovo")[1], "errores dentro de la rama")
atributo2 = ___                                          # el atributo con menos errores dentro de la rama
reglas2, _ = oner(socios_rama, atributo2, "Renovo")
print("Segunda pregunta:", atributo2, reglas2)

# (3) Acierto del árbol en prueba
pred_arbol = [predecir_arbol(f, ganador, reglas_ganador, rama, atributo2, reglas2) for _, f in prueba.iterrows()]
print("Acierto del árbol en prueba:", round(acierto(pred_arbol), 3))

**✅ Compare:** su celda 📊 debe mostrar los **errores de cada rama**, la **rama impura**, la **segunda pregunta** y el **acierto del árbol en prueba**.

### ✍️ Responda

**1. Sus resultados:** ¿Cuál es la rama impura y cuál la segunda pregunta? Ordene ZeroR, OneR y el árbol según su acierto en prueba.

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Escriba el árbol como un **protocolo para el equipo de recepción**: ¿a qué socios hay que contactar antes de que venza su plan, y qué ofrecerles?

_Su respuesta:_

---
# Parte II · Librería online (35 puntos)

**Apriori.** La tabla `pedidos` tiene una fila por producto comprado (`Pedido`, `Producto`). Ya está convertida en **`canastas`**: una lista de **320 conjuntos**, uno por pedido. Por ejemplo, `{'Témpera', 'Pincel', 'Cuaderno'}`. Los 6 productos están en `PRODUCTOS`.

📄 **Antes de la tarea 4, lea la guía de apoyo «soporte(), confianza() y lift()».**

**Regresión.** La tabla `semanas` tiene 36 semanas con `Correos` (los correos promocionales enviados esa semana, en miles) y `Pedidos` (los pedidos de esa semana).

## Tarea 4 · Soporte: ¿qué se vende lo suficiente? (8 puntos)

**📖 Contexto.** La librería quiere recomendar productos («Otros clientes también llevaron…»). Primero mira **qué tan seguido** se vende cada uno.

> **💡 Concepto clave.** El **soporte** es la proporción de pedidos que contienen un producto (o varios juntos). Si un producto casi no se vende (menos de 10 %), no vale la pena buscar reglas con él.

### 🎯 Qué hacer

1. Lea la guía «soporte(), confianza() y lift()».
2. Complete `soporte()` siguiendo las pistas y ejecute la prueba 🧪.
3. Complete el `___` de la celda 📊.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| El **soporte de los 6 productos** y cuáles llegan al 10 %. | Muestra **qué tan populares** son los productos. Ojo: un producto que está en casi todos los pedidos puede **engañar** en la tarea 5. |

In [ ]:
# 🧩 Tarea 4 · complete el cuerpo de la(s) función(es)
def soporte(canastas, productos):
    """Soporte: en qué proporción de TODOS los pedidos aparecen juntos los productos dados.

    Recibe:   canastas  → lista de pedidos; cada pedido es un conjunto, p. ej. {"Témpera", "Cuaderno"}
              productos → lista con uno o más productos, p. ej. ["Témpera"] o ["Témpera", "Pincel"]
    Devuelve: un número entre 0 y 1
    Ejemplo:  con 5 pedidos, si "Té" está en 3      →  soporte(canastas, ["Té"]) = 0.6
              si "Té" y "Galleta" están juntos en 2  →  soporte(canastas, ["Té", "Galleta"]) = 0.4
    """
    buscados = set(productos)
    cuantos = 0
    for pedido in canastas:
        # Pista: «buscados <= pedido» es True si TODOS los productos buscados están en el pedido.
        # Si se cumple, sume 1 a cuantos.
        pass
    return ___                                   # cuántos pedidos cumplen, dividido por el total: len(canastas)

In [ ]:
# 🧪 Prueba de la tarea 4 con un ejemplo mínimo: ejecútela sin cambiarla
canastas_mini = [{"Té", "Galleta"}, {"Té", "Galleta", "Miel"}, {"Té"}, {"Miel"}, {"Galleta"}]
print(soporte(canastas_mini, ["Té"]), soporte(canastas_mini, ["Té", "Galleta"]))
# ✅ Debe mostrar: 0.6 0.4

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`0.6 0.4`**: Té está en 3 de los 5 pedidos del ejemplo; Té y Galleta juntos, en 2 de 5.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 4 · complete los ___ y ejecute
for p in PRODUCTOS:
    s = soporte(canastas, [___])                         # el soporte del producto p
    print(p, round(s, 3), "frecuente" if s >= 0.10 else "NO frecuente")

**✅ Compare:** su celda 📊 debe mostrar el **soporte de los 6 productos** y cuáles llegan al 10 %.

### ✍️ Responda

**1. Sus resultados:** ¿Qué producto no es frecuente? ¿Cuál aparece en casi todos los pedidos?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** ¿Vale la pena armar promociones con el producto no frecuente? ¿Por qué?

_Su respuesta:_

## Tarea 5 · Confianza y lift: ¿qué regla sirve para recomendar? (12 puntos)

**📖 Contexto.** Una regla **A → B** dice «quien lleva A también lleva B». La librería quiere saber **cuál usar para recomendar**.

> **💡 Concepto clave.** **Confianza**: de los pedidos con A, qué parte lleva B. **Lift** = confianza / soporte de B. **Lift mayor que 1**: la regla sirve. **Lift cercano a 1 o menor**: no sirve, aunque la confianza sea alta.

### 🎯 Qué hacer

1. Complete `confianza()` y `lift()` (use su `soporte()`) y ejecute la prueba 🧪.
2. Complete los `___` de la celda 📊 para las 3 reglas.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| El **soporte, la confianza y el lift** de 3 reglas: Témpera → Pincel, Pincel → Témpera y Block → Cuaderno. | La confianza dice **qué tan seguido** se cumple la regla; el lift dice si **sirve para recomendar**. |

In [ ]:
# 🧩 Tarea 5 · complete el cuerpo de la(s) función(es)
def confianza(canastas, a, b):
    """Confianza de la regla a → b: de los pedidos que tienen a, qué proporción tiene también b.

    Confianza(a → b) = soporte(a y b) / soporte(a)      (se divide por el producto de la IZQUIERDA)
    Ejemplo:  si 3 pedidos tienen Té y 2 de ellos también Galleta  →  confianza(Té → Galleta) = 2/3
    """
    # confianza(a → b) = soporte de a y b JUNTOS / soporte de a
    # Pista: soporte(canastas, [a, b]) y soporte(canastas, [a])
    return ___


def lift(canastas, a, b):
    """Lift de la regla a → b: cuántas veces más se lleva b cuando se lleva a, comparado con el azar.

    Lift(a → b) = confianza(a → b) / soporte(b)
    Lectura:  mayor que 1 → asociación real · igual a 1 → no hay relación · menor que 1 → se evitan
    Ejemplo:  confianza(Té → Galleta) = 0.667 y Galleta está en el 60 % de los pedidos (0.6)
              →  lift = 0.667 / 0.6 = 1.11
    """
    # lift(a → b) = confianza(a → b) / soporte de b
    # Pista: confianza(canastas, a, b) y soporte(canastas, [b])
    return ___

In [ ]:
# 🧪 Prueba de la tarea 5 con un ejemplo mínimo: ejecútela sin cambiarla
canastas_mini = [{"Té", "Galleta"}, {"Té", "Galleta", "Miel"}, {"Té"}, {"Miel"}, {"Galleta"}]
print(round(confianza(canastas_mini, "Té", "Galleta"), 4), round(lift(canastas_mini, "Té", "Galleta"), 4))
# ✅ Debe mostrar: 0.6667 1.1111

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`0.6667 1.1111`**: de 3 pedidos con Té, 2 llevan Galleta (0,67); Galleta está en 3 de 5 pedidos (0,6) → lift 0,67 / 0,6 = 1,11.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 5 · complete los ___ y ejecute
for a, b in [("Témpera", "Pincel"), ("Pincel", "Témpera"), ("Block", "Cuaderno")]:
    print(a, "→", b,
          "| soporte:", round(soporte(canastas, [a, b]), 3),
          "| confianza:", round(___, 3),                 # use su función confianza
          "| lift:", round(___, 3))                      # use su función lift

**✅ Compare:** su celda 📊 debe mostrar el **soporte, la confianza y el lift** de 3 reglas: Témpera → Pincel, Pincel → Témpera y Block → Cuaderno.

### ✍️ Responda

**1. Sus resultados:** ¿Qué regla tiene **más confianza**? ¿Y **más lift**? ¿Por qué no es la misma?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Si un cliente pone **Block** en el carro, ¿conviene mostrarle Cuaderno? ¿Qué regla usaría para recomendar?

_Su respuesta:_

## Tarea 6 · Regresión: ¿cuántos pedidos traen los correos? (15 puntos)

**📖 Contexto.** La librería anotó, durante 36 semanas, **cuántos correos promocionales envió** (en miles) y **cuántos pedidos** recibió. Quiere saber cuántos pedidos esperar según los correos que envíe.

> **💡 Concepto clave.** La **regresión lineal** dibuja la recta que mejor sigue a los puntos: **pedidos = a + b · correos**. **b** (la pendiente) dice cuántos pedidos más llegan por cada 1.000 correos extra. La recta solo es confiable **dentro del rango** de envíos observados; usarla fuera es **extrapolar**.

### 🎯 Qué hacer

1. No hay función que completar.
2. Complete los `___` de la celda 📊: ajuste el modelo, muestre **a** y **b**, prediga los pedidos con **30 mil correos** y muestre el envío **mínimo y máximo** observado.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Los números **a** y **b** de la recta, la **predicción con 30 mil correos** y el **rango** de envíos observado. | **b** es lo que rinde cada 1.000 correos, en pedidos reales. El rango dice **hasta dónde se puede confiar** en la recta. |

In [ ]:
# 📊 Tarea 6 · complete los ___ y ejecute
X = semanas[["Correos"]]                                 # lo que se usa para predecir
y = semanas["Pedidos"]                                   # lo que se quiere predecir
modelo = LinearRegression().fit(___, ___)
print("a =", round(modelo.intercept_, 1))
print("b =", round(modelo.coef_[0], 2))
print("Pedidos esperados con 30 mil correos:", round(modelo.predict(pd.DataFrame({"Correos": [___]}))[0], 1))
print("Correos observados: de", semanas["Correos"].min(), "a", semanas["Correos"].max(), "mil")

**✅ Compare:** su celda 📊 debe mostrar los números **a** y **b** de la recta, la **predicción con 30 mil correos** y el **rango** de envíos observado.

### ✍️ Responda

**1. Sus resultados:** Escriba la recta con sus números. Explique a la dueña qué significa **b**. ¿Cuántos pedidos se esperan con 30 mil correos?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** La dueña quiere enviar **200 mil correos** en una semana y usar la recta para saber cuántos pedidos tendrá. ¿Qué le diría?

_Su respuesta:_

---
# Parte III · Preuniversitario online (30 puntos)

La tabla `preu` tiene **57 estudiantes** con `Video` (horas a la semana viendo clases grabadas) y `Ejercicios` (horas a la semana resolviendo ejercicios). Ya están en **`puntos`**: una lista de pares `(x, y)`.

Se buscan **3 grupos** con k-means, partiendo de las estrellas **`ESTRELLAS_INICIO` = (1; 1), (9; 2) y (8; 3)**.

**Recuerde:** una **ronda** = **(1) elegir:** cada estudiante va a la estrella más cercana + **(2) moverse:** cada estrella va al promedio de su grupo. Se repite hasta que en una ronda **nadie cambie de grupo**.

## Tarea 7 · Una ronda de k-means: elegir y moverse (12 puntos)

**📖 Contexto.** La coordinadora del preuniversitario quiere formar **3 grupos de estudiantes parecidos** (según horas de video y de ejercicios) para darles un apoyo distinto.

> **💡 Concepto clave.** Cada grupo tiene una **estrella** (su centro). Una **ronda** tiene dos pasos: **elegir** (cada estudiante va a la estrella más cercana) y **moverse** (cada estrella va al promedio de su grupo).

### 🎯 Qué hacer

1. Complete `asignar()` (elegir) y `actualizar()` (moverse) siguiendo sus pistas.
2. Ejecute la prueba 🧪.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Que la prueba 🧪 dé **exactamente** lo indicado. | Estas dos funciones **son una ronda completa**: en la tarea 8 se repiten hasta que los grupos no cambien. |

In [ ]:
# 🧩 Tarea 7 · complete el cuerpo de la(s) función(es)
def asignar(puntos, estrellas):
    """Paso 1 de una ronda (ELEGIR): cada punto elige la estrella más cercana.

    Recibe:   puntos    → lista de pares (x, y), p. ej. [(2, 8), (9, 2), ...]
              estrellas → lista de pares (x, y), una por grupo
    Devuelve: lista de grupos, uno por punto: 0 = primera estrella, 1 = segunda, ...
    Para comparar basta d² = (x − x★)² + (y − y★)², sin raíz.
    Ejemplo:  asignar([(9, 1), (2, 7)], [(1, 8), (8, 0)])  →  [1, 0]
    """
    grupos = []
    for x, y in puntos:
        d2 = [(x - ex) ** 2 + (y - ey) ** 2 for ex, ey in estrellas]   # d² a cada estrella
        # Pista: el grupo es la POSICIÓN de la menor distancia  →  d2.index(min(d2))
        grupos.append(___)
    return grupos


def actualizar(puntos, grupos, estrellas):
    """Paso 2 de una ronda (MOVERSE): cada estrella se va al promedio de los puntos de su grupo.

    Recibe:   puntos, grupos (lo que devuelve asignar) y las estrellas actuales
    Devuelve: lista de nuevas estrellas (x, y). Si un grupo quedara vacío, su estrella no se mueve.
    Ejemplo:  actualizar([(2, 8), (4, 6), (9, 1)], [0, 0, 1], [(0, 10), (10, 0)])  →  [(3.0, 7.0), (9.0, 1.0)]
    """
    nuevas = []
    for k in range(len(estrellas)):
        mios = [p for p, g in zip(puntos, grupos) if g == k]          # los puntos del grupo k
        if mios:
            x_prom = ___            # promedio de las x   (pista: sum(p[0] for p in mios) / len(mios))
            y_prom = ___            # promedio de las y
            nuevas.append((x_prom, y_prom))
        else:
            nuevas.append(estrellas[k])                               # grupo vacío: la estrella no se mueve
    return nuevas

In [ ]:
# 🧪 Prueba de la tarea 7 con un ejemplo mínimo: ejecútela sin cambiarla
puntos_mini = [(2, 8), (1, 9), (4, 6), (9, 2), (8, 1)]
print(asignar(puntos_mini, [(0, 10), (6, 4)]))
# ✅ Debe mostrar: [0, 0, 1, 1, 1]
puntos_mini = [(2, 8), (1, 9), (4, 6), (9, 2), (8, 1)]
nuevas = actualizar(puntos_mini, [0, 0, 1, 1, 1], [(0, 10), (6, 4)])
print([(round(float(x), 2), round(float(y), 2)) for x, y in nuevas])
# ✅ Debe mostrar: [(1.5, 8.5), (7.0, 3.0)]

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`[0, 0, 1, 1, 1]`**: con las estrellas en (0; 10) y (6; 4), los dos primeros puntos quedan más cerca de la primera estrella (grupo 0) y los otros tres de la segunda (grupo 1).
- **`[(1.5, 8.5), (7.0, 3.0)]`**: la primera estrella va al promedio de (2; 8) y (1; 9) = (1,5; 8,5); la segunda, al de (4; 6), (9; 2) y (8; 1) = (7; 3).

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

## Tarea 8 · Repetir rondas y graficarlas (18 puntos)

**📖 Contexto.** Una ronda no basta: al moverse las estrellas, algunos estudiantes quedan más cerca de otra y **cambian de grupo**. Se repite hasta que nadie cambie.

> **💡 Concepto clave.** El algoritmo **termina** cuando en una ronda **nadie cambia de grupo**. Graficar cada ronda muestra cómo las estrellas **llegan al centro de cada grupo**.

### 🎯 Qué hacer

1. Complete los `___` de `graficar_ronda()` y de `kmeans_por_rondas()`: la estructura ya está armada.
2. Ejecute la prueba 🧪.
3. Ejecute la celda 📊: corre los 57 estudiantes y dibuja una gráfica por ronda.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| **Una gráfica por ronda**, cuántos estudiantes **cambian en cada ronda** (la última debe ser 0) y las **3 estrellas finales**. | Las estrellas finales son el **estudiante típico de cada grupo**: con ellas la coordinadora decide qué apoyo dar a cada uno. |

In [ ]:
# 🧩 Tarea 8 · complete el cuerpo de la(s) función(es)
def graficar_ronda(puntos, grupos, estrellas, titulo):
    """Dibuja UNA ronda: los puntos coloreados según su grupo y cada estrella como ★ grande.

    Requisitos de la gráfica:
      • cada grupo con un color distinto, y su estrella del MISMO color, con marcador "*" grande
      • ejes de 0 a 10, con nombre: "Horas de video" (x) y "Horas de ejercicios" (y)
      • el título que se recibe (debe decir el número de ronda)
    No devuelve nada: solo muestra la gráfica (plt.show()).
    """
    colores = ["tab:red", "tab:blue", "tab:green"]
    plt.figure(figsize=(5, 5))
    for k, (ex, ey) in enumerate(estrellas):
        xs = [p[0] for p, g in zip(puntos, grupos) if g == k]         # las x de los puntos del grupo k
        ys = ___                                                      # las y de los puntos del grupo k
        plt.scatter(xs, ys, color=colores[k])                         # dibuja los puntos del grupo k
        plt.scatter([ex], [ey], color=colores[k], marker=___, s=400, edgecolor="black")   # la estrella: marker="*"
    plt.xlim(0, 10)
    plt.ylim(0, 10)
    plt.xlabel(___)                                                   # nombre del eje x
    plt.ylabel(___)                                                   # nombre del eje y
    plt.title(titulo)
    plt.show()


def kmeans_por_rondas(puntos, estrellas_inicio, graficar=True):
    """El CICLO de k-means: repite rondas (elegir + moverse) hasta que en una ronda NADIE cambie.

    En cada ronda:
      1. grupos = asignar(puntos, estrellas)
      2. se cuenta cuántos puntos cambiaron de grupo respecto de la ronda anterior
         (en la ronda 1 no hay con qué comparar: cambios = None)
      3. se guarda la ronda en el historial y, si graficar=True, se grafica con graficar_ronda()
      4. si cambios == 0 → se termina; si no → estrellas = actualizar(...) y se sigue
    Devuelve: (estrellas_finales, grupos_finales, historial)
              historial = lista con un diccionario por ronda: {"ronda": 1, "cambios": None, "estrellas": [...]}
    Ejemplo:  si en la ronda 2 cambian dos puntos y en la 3 ninguno  →  cambios [None, 2, 0]
    """
    estrellas = list(estrellas_inicio)
    anterior = None
    historial = []
    ronda = 0
    while True:
        ronda = ronda + 1
        grupos = ___                       # paso 1, ELEGIR: use asignar(puntos, estrellas)
        if anterior is None:
            cambios = None                 # en la ronda 1 no hay con qué comparar
        else:
            cambios = sum(a != b for a, b in zip(anterior, grupos))   # cuántos cambiaron de grupo
        historial.append({"ronda": ronda, "cambios": cambios, "estrellas": estrellas})
        if graficar:
            graficar_ronda(puntos, grupos, estrellas, f"Ronda {ronda} · cambios: {cambios}")
        if cambios == 0:
            return estrellas, grupos, historial     # nadie cambió: terminó
        estrellas = ___                    # paso 2, MOVERSE: use actualizar(puntos, grupos, estrellas)
        anterior = grupos

In [ ]:
# 🧪 Prueba de la tarea 8 con un ejemplo mínimo: ejecútela sin cambiarla
puntos_mini = [(2, 8), (1, 9), (4, 6), (9, 2), (8, 1)]
estrellas_m, grupos_m, historial_m = kmeans_por_rondas(puntos_mini, [(0, 10), (6, 4)], graficar=False)
print(len(historial_m), [h["cambios"] for h in historial_m])
# ✅ Debe mostrar: 3 [None, 1, 0]

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`3 [None, 1, 0]`**: 3 rondas: en la 1 no hay con qué comparar (None); en la 2 el punto (4; 6) cambia al grupo 0, porque las estrellas se movieron (1 cambio); en la 3 nadie cambia (0) y termina.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 8 · complete los ___ y ejecute
estrellas_fin, grupos_fin, historial = kmeans_por_rondas(puntos, ESTRELLAS_INICIO, graficar=True)
print("Cambios por ronda:", [h["cambios"] for h in historial])
print("Estrellas finales:", [(round(float(x), 2), round(float(y), 2)) for x, y in estrellas_fin])
print("Estudiantes por grupo:", [grupos_fin.count(k) for k in range(3)])

**✅ Compare:** su celda 📊 debe mostrar **Una gráfica por ronda**, cuántos estudiantes **cambian en cada ronda** (la última debe ser 0) y las **3 estrellas finales**.

### ✍️ Responda

**1. Sus resultados:** ¿En cuántas rondas termina y cuántos cambian en cada una? ¿Dónde quedan las estrellas?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Mirando las estrellas finales: ¿qué tipo de estudiante representa cada grupo? Póngales nombre y proponga **un apoyo** para uno.

_Su respuesta:_

---
## ✔️ Para entregar

1. **Entorno de ejecución → Reiniciar y ejecutar todo**: revise que todo corra sin errores.
2. **Archivo → Descargar → Descargar .ipynb**.
3. Súbalo al **Aula Virtual** como `Apellido_Nombre.ipynb`.